# Movie preferences: data preparation, EDA and rating baselines

This is the preliminary mining notebook for the progress report. It uses all published MovieLens ratings and opens the warehouse read-only.

**Reading order:** understand the data, describe users, then compare two simple rating predictions.

H2 asks whether rating habits and genre/tag preferences reveal user groups. H3 asks whether a supervised model can beat global-average and movie-average predictions. This milestone prepares user descriptors and measures the baselines; it does not fit a clustering or supervised model.

The earlier clustering experiment is preserved under `notebooks/archive/`. Tags and tag-genome features are not yet included in this analysis.

## 1. Open the cleaned warehouse

ETL has already checked the ratings. Here we use those cleaned records and do not change the warehouse. Run from the project root, or use the Docker instructions in `MINING_PRELIMINARY.md`.


In [ ]:
from IPython.display import display, Image
import pandas as pd
from mining.preliminary import Session, run_eda, prepare_features, run_baselines, finalize

ctx = Session()
summary = {'ratings': 'Rating events', 'rating_users': 'Users with ratings',
           'rated_movies': 'Movies with ratings', 'first_rating': 'First rating (UTC)',
           'last_rating': 'Last rating (UTC)'}
display(pd.DataFrame([ctx.before])[list(summary)].rename(columns=summary))


## 2. Explore the ratings (EDA)

EDA means **exploratory data analysis**: use counts, averages and charts to understand the data before building models.

- Rating distribution: which scores occur most often?
- Activity: how many ratings were submitted over time and per user/movie?
- Genre exposure: how are ratings shared across movie genres?

A rating is a recorded score, not proof of a movie view. For two genres, each receives half a rating in the weighted genre chart. This keeps total allocated ratings equal to the original total.


In [ ]:
eda = run_eda(ctx)
print(f"Ratings analyzed: {eda['ratings']:,}")
print(f"Average score: {eda['mean_rating']:.2f} out of 5")
print(f"Most common score: {eda['most_common_rating']:.1f}")
print(f"Median ratings per user: {eda['user_activity_quantiles']['0.5']:.0f}")
display(Image(filename=str(ctx.out / 'eda_rating_distribution.png')))


### Activity over time and across users/movies

The first chart shows monthly rating activity. The second compares how many ratings users submit and movies receive. Its axes use a log scale to show both small and very large counts; this is a chart scale, not a mining model.


In [ ]:
display(Image(filename=str(ctx.out / 'eda_activity.png')))


### Movie genres

Genre exposure counts rated movies through the weighted genre bridge. A large count means many rating events involve that genre. It does not, by itself, prove stronger liking for that genre.


In [ ]:
display(Image(filename=str(ctx.out / 'eda_genres.png')))


## 3. Prepare simple user descriptors

A feature is a value that describes a record. Here **one row represents one user**.

| Descriptor | Meaning |
|---|---|
| Rating count | Number of ratings submitted |
| Mean rating | Average score given |
| Rating standard deviation | How much the scores vary |
| High-rating share | Fraction of scores that are at least 4 |
| Activity span | Days between the first and last rating |
| Genre shares | Fraction of weighted rating activity in each genre |

We prepare five behavior descriptors and 20 genre shares. We keep their original units; this notebook does not scale features or fit clusters. A zero genre share means no observed ratings in that genre, not dislike.


In [ ]:
users = prepare_features(ctx, descriptive_only=True)
print(f"Users described: {len(users):,}")
print(f"Prepared descriptors per user: {len(ctx.features)}")
columns = {'rating_count':'Ratings given', 'mean_rating':'Average score',
           'rating_std':'Score variation', 'high_rating_share':'Share scoring at least 4',
           'activity_span_days':'Activity span (days)'}
display(users[list(columns)].head().rename(columns=columns).round(3))


### Feature strategy for H2 and H3

**H2:** use rating habits and genre shares to study similar-user groups later.

**H3:** prepare user/movie history using only ratings available before the rating being predicted. A target rating must not enter its own features. The full-history table above describes users and must not be used directly for prediction.

Tags and tag-genome features remain future work. Current TMDb popularity and vote details are excluded from these baselines because they were not observed at the historical rating dates.

## 4. Compare two simple rating baselines

A baseline is a simple prediction method used as a reference for a later model.

| Method | Prediction |
|---|---|
| Global average | Average score across training ratings |
| Movie average | That movie's average training score; use the global average if it has no training ratings |

We split ratings by time: approximately **70% training, 15% validation and 15% test**. Earlier ratings come first, and equal timestamps stay together. Training calculates the averages. Validation supports later model development. Test ratings measure performance on later events.

Both predictions use **training ratings only**. This prevents evaluation answers from entering the prediction averages.


In [ ]:
baselines = run_baselines(ctx)
split_table = pd.DataFrame(baselines['splits'])
split_table['split'] = pd.Categorical(split_table['split'], ['train', 'validation', 'test'], ordered=True)
display(split_table.sort_values('split')[['split','ratings']].rename(
    columns={'split':'Dataset','ratings':'Rating events'}))
print(f"Global-average prediction: {baselines['training_global_mean']:.3f}")


### Read the prediction errors

**MAE** is the average absolute difference between a prediction and the real score. Predicting 3.5 for a real score of 4.0 gives an absolute error of 0.5.

**RMSE** gives larger mistakes more weight. Both measures use rating-score units; **lower is better**. The two methods below are evaluated on exactly the same test ratings.


In [ ]:
test = pd.DataFrame(baselines['metrics'])
test = test.loc[test['split'] == 'test'].copy()
test['model'] = test['model'].replace({'global_average':'Global average',
                                     'movie_average':'Movie average + fallback'})
display(test[['model','rmse','mae']].rename(
    columns={'model':'Baseline','rmse':'Test RMSE','mae':'Test MAE'}).round(4))
fallback = test.loc[test['model'] == 'Movie average + fallback', 'global_fallback_events'].iloc[0]
print(f"Test events using the global fallback for unseen movies: {fallback:,}")
display(Image(filename=str(ctx.out / 'h3_baseline_errors.png')))


## 5. Save tables, charts and the report section

Save the EDA, prepared descriptors, time split and baseline errors. The run records the source ETL and software versions. A final count/sum/date comparison checks that the read-only warehouse is unchanged.


In [ ]:
result = finalize(ctx, eda, None, baselines)
print("Run status:", result['status'])
print("Warehouse summary unchanged:", result['warehouse_signature_unchanged'])
print("Saved numerical results:", ctx.out)
print("Simple report section: reports/results/mining_milestone/REPORT_SECTION.md")


## What this milestone shows

We have explored the ratings, prepared user descriptors and established two prediction references.

No clustering model is fitted here, so H2 remains a research question. No supervised model is evaluated here, so baseline performance does not confirm H3.

The next mining work is to choose class-appropriate models and compare them fairly. Keep the original H2/H3 hypotheses; explain any feature limitations when presenting results.
